# iSWAP Module

The iSWAP is a plate transport gripper arm on the Hamilton STAR(let). After {meth}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR.setup`, it is available as `star.iswap` — an {class}`~pylabrobot.capabilities.arms.arm.OrientableArm` whose backend is an {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend`.

This notebook covers low-level iSWAP control: parking, gripper, rotations, slow movement, and manual teaching/calibration. For high-level plate movement, use `star.iswap.move_resource(plate, destination)`.

## Setup

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARLetDeck

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

After `setup()`, `star.iswap` is an {class}`~pylabrobot.capabilities.arms.arm.OrientableArm` if the hardware is installed, or `None` if it is not. The low-level backend is `star.iswap.backend`, which is an {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend`.

## Common tasks

### Parking

Park the iSWAP using {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.park`. You can optionally pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.ParkParams` to control the minimum traverse height.

In [ ]:
await star.iswap.backend.park()

### Opening the gripper

Open the iSWAP gripper using {meth}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm.open_gripper`. **Warning**: this will release any object that is gripped. Useful for error recovery.

In [ ]:
# open all the way
await star.iswap.open_gripper(gripper_width=130)

In [ ]:
# open partially to 90 mm
await star.iswap.open_gripper(gripper_width=90)

### Closing the gripper

Close the iSWAP gripper using {meth}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm.close_gripper`. This will throw an error if there is no object to grip. You can optionally pass {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.CloseGripperParams` to control grip strength and plate width tolerance.

In [ ]:
await star.iswap.close_gripper(gripper_width=85)

## Rotations

You can rotate the iSWAP to 12 predefined positions using {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.rotate`.

The positions and their corresponding integer specifications are shown visually here:

![iSWAP positions](img/iswap_positions.png)

The `rotate` method moves the wrist drive and the rotation drive simultaneously in one smooth motion. It takes a parameter for the rotation drive ({class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.RotationDriveOrientation`), and the final `grip_direction` ({class}`~pylabrobot.capabilities.arms.standard.GripDirection`) of the iSWAP. The `grip_direction` is the same parameter used by high-level plate movement and is with respect to the deck. This is easier than controlling the rotation drive, which is with respect to the wrist drive.

For example, to extend the iSWAP fully to the left, call `rotate(rotation_drive=iSWAPBackend.RotationDriveOrientation.LEFT, grip_direction=GripDirection.RIGHT)`. `GripDirection.RIGHT` means the gripper will be gripping the object from the right hand side, so the gripper fingers point left with respect to the deck. Compare this to `rotate(rotation_drive=iSWAPBackend.RotationDriveOrientation.RIGHT, grip_direction=GripDirection.RIGHT)`, where the gripper fingers still point left, but the rotation drive points right and the wrist drive is in "REVERSE" orientation (folded up).

Moving between two positions with the same `grip_direction` while changing the rotation drive will keep the plate pointing in one direction. The internal motion planner on the STAR automatically adjusts the wrist drive.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.iswap import iSWAPBackend
from pylabrobot.capabilities.arms.standard import GripDirection

await star.iswap.backend.rotate(
  rotation_drive=iSWAPBackend.RotationDriveOrientation.LEFT,
  grip_direction=GripDirection.RIGHT,
)

### Controlling the wrist and rotation drive individually

You can also control the wrist (T-drive) and rotation drive (W-drive) individually using {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.rotate_wrist` and {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.rotate_rotation_drive` respectively. Make sure you have enough space (you can use {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.move_relative_y` to create clearance).

In [ ]:
import random

rotation_drive = random.choice([
  iSWAPBackend.RotationDriveOrientation.LEFT,
  iSWAPBackend.RotationDriveOrientation.RIGHT,
  iSWAPBackend.RotationDriveOrientation.FRONT,
])
wrist_drive = random.choice([
  iSWAPBackend.WristDriveOrientation.LEFT,
  iSWAPBackend.WristDriveOrientation.RIGHT,
  iSWAPBackend.WristDriveOrientation.STRAIGHT,
  iSWAPBackend.WristDriveOrientation.REVERSE,
])
await star.iswap.backend.rotate_rotation_drive(rotation_drive)
await star.iswap.backend.rotate_wrist(wrist_drive)

## Slow movement

You can make the iSWAP move more slowly during sensitive operations using the {meth}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.slow` context manager. This is useful when you want to avoid splashing or other disturbances.

In [ ]:
async with star.iswap.backend.slow():
  await star.iswap.move_resource(plate, destination)

## Manual movement (teaching / calibration)

### 1. Clear the Y range

For safety, move the other components as far away as possible before teaching. This is done using the firmware command `C0FY`, which is available on the legacy backend.

```{note}
`position_components_for_free_iswap_y_range` is currently only available via the legacy `STARBackend`. It has not yet been migrated to the new driver.
```

### 2. Set rotation

Move the iSWAP wrist and rotation drive to the correct orientation as [explained above](#rotations). Be careful to move the iSWAP to a position where it does not hit any other components.

### 3. Absolute movement

Use the following methods to move the iSWAP to absolute X, Y and Z positions. All units are in mm.

In [ ]:
await star.iswap.backend.move_x(x=200)

In [ ]:
await star.iswap.backend.move_y(y=200)

In [ ]:
await star.iswap.backend.move_z(z=200)

### 4. Computing plate position from calibration

The x, y, and z values refer to the **center** of the iSWAP gripper, making them agnostic to plate size. In PLR, however, all locations are with respect to the left-front-bottom (LFB) of the plate. To convert from the calibrated center to LFB, subtract the plate's center-bottom anchor:

In [ ]:
from pylabrobot.resources import Coordinate

x, y, z = 200, 200, 200  # calibrated center position
calibrated_position = Coordinate(x, y, z)
plate_lfb_absolute = calibrated_position - plate.get_anchor("c", "c", "b")

The plate's LFB is now in absolute coordinates. If the plate is a child of some parent resource, compute the relative location by subtracting the parent's absolute position:

In [ ]:
parent_absolute = parent.get_location_wrt(deck)
plate_relative = plate_lfb_absolute - parent_absolute

Use this with `parent.assign_child_resource(plate, location=plate_relative)` to place the plate at the calibrated position.

### 5. Relative movements

You can also move the iSWAP relative to its current position. All units are in mm. These refer to the center of the gripper.

In [ ]:
await star.iswap.backend.move_relative_x(step_size=10)

In [ ]:
await star.iswap.backend.move_relative_y(step_size=10)

In [ ]:
await star.iswap.backend.move_relative_z(step_size=10)

## Backend parameters

For fine-grained control, many iSWAP operations accept `backend_params`. The available parameter classes are:

- {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.ParkParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.CloseGripperParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.PickUpParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.DropParams`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.iswap.iSWAPBackend.MoveToLocationParams`

## Teardown

In [ ]:
await star.stop()